[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_05_knn_naive_bayes.ipynb)

# S07 · 05 — k vecinos (k-NN) y Naive Bayes

🏠 Cuaderno para casa · ⏱ ≈ 40 min · Sprint 7 · M3 (UD 3.5)

**Contexto TurisData.** Tenemos ya la regresión logística como referencia para predecir cancelaciones. Probamos dos algoritmos con una filosofía muy distinta:

- **k-NN:** «dime con quién se parece esta reserva y te digo qué hará» (razona por **analogía**).
- **Naive Bayes:** «combino la probabilidad de cada dato por separado» (razona con **probabilidades**).

**Este cuaderno tiene 4 ejercicios y un mini-reto** y es **solo para casa**: no tiene versión «en clase». Es refuerzo: el reto del sprint se hace en clase y no depende de este cuaderno. Hazlo en casa, a tu ritmo y por orden.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)


def prep_onehot():
    """Imputar + escalar (numéricas) e imputar + one-hot denso (categóricas)."""
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
    ])
print("Listo ✔")

### 👀 A2 · EJEMPLO — ¿Qué variable manda en la distancia?

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué porcentaje de la distancia aporta `antelacion_dias` sin escalar y escalada?
2. Sin escalar, ¿qué variables deciden casi toda la distancia?
3. ¿Qué pasaría si k-NN no escalara y `cliente_repetidor` fuera muy importante?

In [ ]:
# ¿Qué variable manda en la distancia? Comparamos qué % de la distancia² entre pares aporta cada una
rng = np.random.default_rng(SEMILLA)
numericas = df[columnas_num].copy()
def reparto_distancia(tabla, n_pares=3000):
    i = rng.integers(0, len(tabla), n_pares)
    j = rng.integers(0, len(tabla), n_pares)
    cuadrados = (tabla.iloc[i].to_numpy() - tabla.iloc[j].to_numpy()) ** 2
    return pd.Series(cuadrados.mean(axis=0) / cuadrados.mean(axis=0).sum(), index=tabla.columns)

sin_escalar = reparto_distancia(numericas)
escalado = reparto_distancia(pd.DataFrame(StandardScaler().fit_transform(numericas), columns=columnas_num))
pd.DataFrame({"sin escalar": sin_escalar, "escaladas": escalado}).mul(100).round(1)

### ✏️ A3 · EJERCICIO 1 de 4 — Cuánto pesa la antelación en la distancia

**Tu turno:** Guarda en `pct_antelacion_sin_escalar` y `pct_antelacion_escalada` el % de la distancia que aporta `antelacion_dias` (de las Series `sin_escalar` y `escalado`, en tanto por ciento). ¿Qué pasa con las demás variables cuando no se escala?

**Debe salir:** `Sin escalar, la antelación decide el 77 % de la distancia; escalada, el 11 %.` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Cuánto valen `pct_antelacion_sin_escalar` y `pct_antelacion_escalada`?
2. Si todas las variables aportaran lo mismo, ¿qué porcentaje correspondería a cada una?
3. ¿Qué demuestra el contraste entre 77 % y 11 %?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: sin_escalar["antelacion_dias"] * 100
pct_antelacion_sin_escalar = ...
pct_antelacion_escalada = ...

assert pct_antelacion_sin_escalar > 60, "Sin escalar, la antelación debería dominar la distancia (más del 60 %)"
assert pct_antelacion_escalada < 25, "Escaladas, cada variable debería aportar una parte parecida"
print(f"Sin escalar, la antelación decide el {pct_antelacion_sin_escalar:.0f} % de la distancia; escalada, el {pct_antelacion_escalada:.0f} %.")
print("✔ Ejercicio 1 correcto")

### ✏️ A4 · EJERCICIO 2 de 4 — Elegir k con validación cruzada

**Tu turno:** Para cada `k` de la lista `ks`, calcula el AUC medio en validación cruzada (5 folds, `scoring="roc_auc"`) de un `Pipeline` con `prep_onehot()` y `KNeighborsClassifier(n_neighbors=k)`. Guarda las medias en la lista `auc_por_k` y el mejor valor de `k` en `mejor_k`.

**Debe salir:** el AUC de cada k (de `k = 1 → AUC 0.585` a `k = 401 → AUC 0.740`), `Mejor k = 201`, un gráfico y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Qué `mejor_k` sale y qué AUC tiene?
2. ¿Por qué k = 1 funciona tan mal (AUC ≈ 0,58)?
3. ¿Qué se pierde al subir mucho k?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
ks = [1, 5, 15, 31, 61, 101, 201, 401]
# pista: un for sobre ks con cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc").mean()
auc_por_k = ...
mejor_k = ...

assert len(auc_por_k) == len(ks), "Una media por cada k"
assert mejor_k >= 31, "Con estos datos ruidosos, los k grandes funcionan mejor (≥ 31)"
assert auc_por_k[0] < max(auc_por_k) - 0.05, "k = 1 debería ser claramente peor (sobreajuste)"

plt.figure(figsize=(7, 3.8))
plt.plot(ks, auc_por_k, marker="o", color="#2a6f97")
plt.xscale("log")
plt.axvline(mejor_k, color="#e63946", ls="--", label=f"Mejor k = {mejor_k}")
plt.title("k-NN: AUC según el número de vecinos")
plt.xlabel("k (escala logarítmica)")
plt.ylabel("AUC medio en validación cruzada")
plt.legend()
plt.show()
for k, a in zip(ks, auc_por_k):
    print(f"k = {k:>3} → AUC {a:.3f}")
print(f"Mejor k = {mejor_k}")
print("✔ Ejercicio 2 correcto")

### ✏️ A5 · EJERCICIO 3 de 4 — Naive Bayes con variables numéricas (GaussianNB)

**Tu turno:** Crea `nb_gauss`, un `Pipeline` con `prep_onehot()` y `GaussianNB()`. Guarda en `auc_nb_gauss` su AUC medio en validación cruzada (mismo `cv`).

**Debe salir:** `AUC GaussianNB: 0.719` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto vale `auc_nb_gauss`?
2. ¿Qué supone Naive Bayes que es «ingenuo»?
3. ¿Por qué pasa por `prep_onehot()` si Naive Bayes no necesita escalar?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: Pipeline([("prep", prep_onehot()), ("nb", GaussianNB())]) y cross_val_score(..., scoring="roc_auc")
nb_gauss = ...
auc_nb_gauss = ...

assert 0.65 < auc_nb_gauss < 0.85, "El AUC de GaussianNB debería estar entre 0,65 y 0,85"
print(f"AUC GaussianNB: {auc_nb_gauss:.3f}")
print("✔ Ejercicio 3 correcto")

### ✏️ A6 · EJERCICIO 4 de 4 — Naive Bayes con variables discretizadas (CategoricalNB)

**Tu turno:** `CategoricalNB` necesita números enteros. Crea `nb_cat`, un `Pipeline` con: - las numéricas → imputar mediana y `KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")`, - las categóricas → imputar moda y `OrdinalEncoder()`, - `CategoricalNB(min_categories=5)`. Guarda en `auc_nb_cat` su AUC medio en validación cruzada.

**Debe salir:** `AUC GaussianNB: 0.719 · CategoricalNB: 0.742` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Qué AUC da `CategoricalNB`, y cuál es mejor, él o `GaussianNB`?
2. ¿Por qué hay que discretizar las numéricas para `CategoricalNB`?
3. ¿Qué significa `strategy="quantile"`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: ColumnTransformer con dos ramas (num: imputar mediana + KBinsDiscretizer; cat: imputar moda + OrdinalEncoder) y CategoricalNB al final
prep_cat = ...
nb_cat = ...
auc_nb_cat = ...

assert 0.65 < auc_nb_cat < 0.85, "El AUC de CategoricalNB debería estar entre 0,65 y 0,85"
print(f"AUC GaussianNB: {auc_nb_gauss:.3f} · CategoricalNB: {auc_nb_cat:.3f}")
print("✔ Ejercicio 4 correcto")

### ✏️ A7 · MINI-RETO — Comparar los cuatro modelos

**Tu turno:** Crea la tabla `tabla` con una fila por modelo (`"logistica"`, `"knn"`, `"nb_gauss"`, `"nb_cat"`) y las columnas: - `auc_cv`: AUC medio (validación cruzada), - `segundos_ajuste`: tiempo en ajustar con todo `X_train` (`time.perf_counter()`), - `segundos_predecir`: tiempo en predecir las probabilidades de `X_test`. Usa `LogisticRegression(max_iter=1000)` y el mejor `k` de antes. Después, en `recomendacion`, escribe tu elección en 2 frases razonando con las tres columnas y con la **explicabilidad**.

**Debe salir:** la tabla `tabla`, tu `recomendacion` impresa y ✔ Mini-reto correcto (≈ logística 0,757 · k-NN 0,740 · nb_gauss 0,719 · nb_cat 0,742)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué modelo tiene el mejor AUC y cuál el peor?
2. ¿Qué modelo tarda más en predecir y cuántas veces más que los demás?
3. ¿Qué modelo recomendarías y por qué? (razona con las tres columnas y la explicabilidad)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
candidatos = {
    "logistica": Pipeline([("prep", prep_onehot()), ("clf", LogisticRegression(max_iter=1000))]),
    "knn": Pipeline([("prep", prep_onehot()), ("clf", KNeighborsClassifier(n_neighbors=mejor_k))]),
    "nb_gauss": nb_gauss,
    "nb_cat": nb_cat,
}
# pista: por cada candidato: cross_val_score; luego mide el tiempo de fit y de predict_proba con time.perf_counter()
tabla = ...
recomendacion = ...

assert list(tabla.index) == ["logistica", "knn", "nb_gauss", "nb_cat"], "Filas en el orden pedido"
assert {"auc_cv", "segundos_ajuste", "segundos_predecir"} <= set(tabla.columns)
assert tabla["auc_cv"].between(0.6, 0.9).all(), "AUC entre 0,6 y 0,9 en todos"
assert isinstance(recomendacion, str) and len(recomendacion) > 40, "Escribe una recomendación razonada"
display(tabla)
print(recomendacion)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, k-NN, los dos Naive Bayes, la regresión logística, el preprocesado y la validación cruzada.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `reservas_turisdata.csv` (6000 reservas) y crea `temporada_alta` (1 si la llegada es en enero, febrero, marzo, julio, agosto o diciembre).
    - Define `X` (9 variables numéricas y 6 categóricas), `y` (si la reserva se cancela) y `cv` (validación cruzada de 5 folds, estratificada y con semilla).
    - Define `prep_onehot()`, que fabrica el preprocesado: imputar y escalar las numéricas, imputar y pasar a *one-hot* las categóricas.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. k-NN: predecir mirando a los vecinos

#### A2 · Ejemplo — ¿Qué variable manda en la distancia?

**Idea.** Para clasificar una reserva nueva, k-NN:
1. calcula su **distancia** a todas las reservas conocidas,
2. se queda con las **k más cercanas** (sus «vecinas»),
3. predice la clase **más frecuente entre ellas** (y la probabilidad, la proporción).

No hay «entrenamiento» real: el modelo **guarda los datos** y trabaja al predecir (por eso puede ser lento con muchos datos). Y como todo depende de **distancias**, una variable con valores grandes (`antelacion_dias`, de 0 a 400) aplasta a las pequeñas (`cliente_repetidor`, 0 o 1). **Escalar es obligatorio.**

- **Qué hace:**
    - `reparto_distancia(tabla)` elige 3000 pares de reservas al azar y calcula qué parte de la distancia² entre ellas aporta cada variable (en tanto por uno).
    - `sin_escalar` lo calcula con los datos tal cual; `escalado`, tras pasarlos por `StandardScaler` (media 0 y desviación 1).
    - La última línea junta las dos Series en una tabla (en %) y es lo que se muestra.
- **Qué debe salir:**
    - Una tabla con 9 filas y dos columnas. Sin escalar: `antelacion_dias` 76,6 %, `precio_noche` 23,2 % y todas las demás ≈ 0. Escaladas: todas entre 10,5 % y 11,5 % (≈ 1/9 cada una).
- **Cómo leerlo:**
    - Mira la columna «sin escalar»: **dos variables se reparten casi toda la distancia** y el resto no cuenta. Con la columna «escaladas» cada variable pesa lo mismo.
- **Si falla:**
    - `NameError: name 'columnas_num' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué porcentaje de la distancia aporta `antelacion_dias` sin escalar y escalada?* → **≈ 77 %** sin escalar y **≈ 11 %** escalada.
2. *Sin escalar, ¿qué variables deciden casi toda la distancia?* → **`antelacion_dias` (77 %) y `precio_noche` (23 %)**: son las de valores más grandes. Las demás aportan menos del 1 % entre todas.
3. *¿Qué pasaría si k-NN no escalara y `cliente_repetidor` fuera muy importante?* → Que el modelo **no se enteraría**: esa variable (0 o 1) no mueve la distancia frente a una antelación de cientos de días. Por eso el escalado es obligatorio en k-NN.

</details>

#### A3 · Ejercicio 1 — Cuánto pesa la antelación en la distancia

- **Qué hace:**
    - `sin_escalar` y `escalado` son Series (una por variable) con la parte de la distancia que aporta cada una, en tanto por uno.
    - `sin_escalar["antelacion_dias"]` toma el valor de esa variable y `* 100` lo pasa a porcentaje; igual con `escalado`.
- **Qué debe salir:**
    - `Sin escalar, la antelación decide el 77 % de la distancia; escalada, el 11 %.`
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `Sin escalar, la antelación debería dominar la distancia (más del 60 %)` → has cogido otra variable o no has multiplicado por 100 (te sale 0,77 y no 77).
    - `Escaladas, cada variable debería aportar una parte parecida` → has usado `sin_escalar` también en la segunda línea: la segunda es `escalado`.
    - `NameError: name 'sin_escalar' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `pct_antelacion_sin_escalar` y `pct_antelacion_escalada`?* → **≈ 76,6** y **≈ 11,4**.
2. *Si todas las variables aportaran lo mismo, ¿qué porcentaje correspondería a cada una?* → **1/9 ≈ 11,1 %**, porque hay 9 variables numéricas. Escalada, la antelación (11,4 %) se queda en ese valor.
3. *¿Qué demuestra el contraste entre 77 % y 11 %?* → Que **el escalado reparte el peso por igual** entre las variables: antes decidían dos; después deciden las nueve.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
pct_antelacion_sin_escalar = sin_escalar["antelacion_dias"] * 100
pct_antelacion_escalada = escalado["antelacion_dias"] * 100
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Elegir k

#### A4 · Ejercicio 2 — Elegir k con validación cruzada

- `k` **pequeño** (1, 3): el modelo copia el ruido de los vecinos (**alta varianza**, sobreajuste).
- `k` **grande**: promedia demasiado y suaviza en exceso (**alto sesgo**).

Se elige con validación cruzada. Usamos el **AUC** para evaluar.

- **Qué hace:**
    - `for k in ks:` recorre los valores de `k`. Para cada uno crea un pipeline con el preprocesado y `KNeighborsClassifier(n_neighbors=k)`.
    - `cross_val_score(..., cv=cv, scoring="roc_auc")` devuelve el AUC en cada uno de los 5 folds; `.mean()` lo promedia. Se añade a `auc_por_k`.
    - `np.argmax(auc_por_k)` da la posición del mayor AUC y `ks[...]` el `k` correspondiente (`mejor_k`).
    - Las líneas que siguen (ya hechas) dibujan el AUC frente a `k` (eje logarítmico) y marcan el mejor `k`. Esta celda tarda unos segundos.
- **Qué debe salir:**
    - Una línea por cada `k` (`k =   1 → AUC 0.585` … `k = 401 → AUC 0.740`) y `Mejor k = 201`.
    - `✔ Ejercicio 2 correcto` y un gráfico: el AUC sube de ≈ 0,58 (k = 1) a ≈ 0,74 (k = 201 y 401) y se estabiliza. La línea roja marca `Mejor k = 201`.
    - `auc_por_k` ≈ `[0.585, 0.645, 0.690, 0.707, 0.722, 0.732, 0.740, 0.740]`.
- **Cómo leerlo:**
    - La curva sube deprisa al principio y **se aplana a partir de k ≈ 100–200**: más vecinos ya no mejoran. Con k = 1 el modelo está sobreajustado (casi no mejora al azar).
- **Si falla:**
    - `Una media por cada k` → `auc_por_k` no tiene 8 valores (¿has hecho `append` fuera del `for`?).
    - `Con estos datos ruidosos, los k grandes funcionan mejor (≥ 31)` → `mejor_k` no es `ks[np.argmax(auc_por_k)]` (¿has cogido `max` en vez de su posición?).
    - `k = 1 debería ser claramente peor (sobreajuste)` → no has usado `scoring="roc_auc"` o el preprocesado sin escalar.
    - `NameError: name 'cv' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué `mejor_k` sale y qué AUC tiene?* → **201**, con AUC ≈ **0,740** (0,7399; con k = 401 sale 0,7396, prácticamente igual).
2. *¿Por qué k = 1 funciona tan mal (AUC ≈ 0,58)?* → Porque copia la clase de **un único vecino**, y con datos ruidosos ese vecino se equivoca a menudo: sobreajuste (alta varianza).
3. *¿Qué se pierde al subir mucho k?* → Que el modelo se vuelve demasiado «promediador» (alto sesgo). Aquí la curva se aplana (0,7399 → 0,7396) y subir más ya no ayuda: lo razonable es el `k` más pequeño que ya rinde bien.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
ks = [1, 5, 15, 31, 61, 101, 201, 401]
auc_por_k = []
for k in ks:
    modelo_k = Pipeline([("prep", prep_onehot()), ("knn", KNeighborsClassifier(n_neighbors=k))])
    auc_por_k.append(cross_val_score(modelo_k, X, y, cv=cv, scoring="roc_auc").mean())
mejor_k = ks[int(np.argmax(auc_por_k))]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Naive Bayes: probabilidades multiplicadas

#### A5 · Ejercicio 3 — Naive Bayes con variables numéricas (GaussianNB)

**Idea.** El teorema de Bayes (ya lo has usado en el Sprint 5) permite calcular `P(cancela | datos de la reserva)`. Naive Bayes hace una simplificación «ingenua»: **supone que las variables son independientes entre sí dado el resultado**, así que basta **multiplicar** la probabilidad que cada dato aporta por separado:

`P(cancela | datos) ∝ P(cancela) · P(antelación | cancela) · P(tarifa | cancela) · …`

Es falso (la antelación y el precio pueden estar relacionados), pero **funciona sorprendentemente bien**, entrena en milisegundos y necesita pocos datos. Sus probabilidades, en cambio, suelen estar mal calibradas (demasiado extremas). Variantes según el tipo de dato:
- **`GaussianNB`:** variables numéricas (supone una campana de Gauss por clase).
- **`CategoricalNB`:** variables categóricas (cuenta frecuencias). Con numéricas, se **discretizan** en tramos.

- **Qué hace:**
    - `Pipeline([("prep", prep_onehot()), ("nb", GaussianNB())])` encadena el preprocesado (escalar y *one-hot*) con el clasificador `GaussianNB` (`nb_gauss`).
    - `cross_val_score(nb_gauss, X, y, cv=cv, scoring="roc_auc").mean()` da su AUC medio en los 5 folds (`auc_nb_gauss`).
- **Qué debe salir:**
    - `AUC GaussianNB: 0.719`.
    - `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `El AUC de GaussianNB debería estar entre 0,65 y 0,85` → `auc_nb_gauss` no sale del pipeline con `GaussianNB` y `scoring="roc_auc"` (¿has usado la exactitud?).
    - `NameError: name 'prep_onehot' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale `auc_nb_gauss`?* → **≈ 0,72** (0,719).
2. *¿Qué supone Naive Bayes que es «ingenuo»?* → Que las variables son **independientes entre sí** dado el resultado, y por eso multiplica las probabilidades de cada una por separado.
3. *¿Por qué pasa por `prep_onehot()` si Naive Bayes no necesita escalar?* → Porque el preprocesado aquí sirve para **convertir las categóricas en números** (*one-hot*) y para imputar los huecos; el escalado no le estorba. Es el mismo preprocesado que usa k-NN, y así se comparan en igualdad de condiciones.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
nb_gauss = Pipeline([("prep", prep_onehot()), ("nb", GaussianNB())])
auc_nb_gauss = cross_val_score(nb_gauss, X, y, cv=cv, scoring="roc_auc").mean()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejercicio 4 — Naive Bayes con variables discretizadas (CategoricalNB)

- **Qué hace:**
    - `prep_cat` es un `ColumnTransformer` con dos ramas: las numéricas se imputan con la mediana y se **discretizan** en 5 tramos con el mismo número de reservas cada uno (`strategy="quantile"`); las categóricas se imputan con la moda y se pasan a enteros con `OrdinalEncoder`.
    - `CategoricalNB(min_categories=5)` es el clasificador, que **cuenta frecuencias** de cada categoría; `min_categories=5` evita errores si un tramo no aparece en un fold.
    - `nb_cat` junta ambas cosas en un `Pipeline` y `auc_nb_cat` es su AUC medio en validación cruzada.
- **Qué debe salir:**
    - `AUC GaussianNB: 0.719 · CategoricalNB: 0.742`.
    - `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - `CategoricalNB` (0,742) supera algo a `GaussianNB` (0,719): al discretizar, captura relaciones que no son una «campana de Gauss».
- **Si falla:**
    - `El AUC de CategoricalNB debería estar entre 0,65 y 0,85` → algún paso está mal: comprueba que las numéricas pasan por `KBinsDiscretizer(encode="ordinal")` y que el clasificador es `CategoricalNB`.
    - `ValueError: ... Negative values in data` o error de categorías → `CategoricalNB` necesita **enteros ≥ 0**: usa `encode="ordinal"` y `OrdinalEncoder()`, no *one-hot*.
    - `NameError: name 'auc_nb_gauss' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC da `CategoricalNB`, y cuál es mejor, él o `GaussianNB`?* → **0,742** frente a **0,719**: gana `CategoricalNB`.
2. *¿Por qué hay que discretizar las numéricas para `CategoricalNB`?* → Porque `CategoricalNB` **cuenta frecuencias de categorías** y necesita valores enteros: se parte cada numérica en 5 tramos (0, 1, 2, 3, 4).
3. *¿Qué significa `strategy="quantile"`?* → Que los 5 tramos tienen **aproximadamente el mismo número de reservas** (cada uno el 20 %), en vez de tener la misma anchura.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
prep_cat = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("bins", KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile"))]), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ord", OrdinalEncoder())]), columnas_cat),
])
nb_cat = Pipeline([("prep", prep_cat), ("nb", CategoricalNB(min_categories=5))])
auc_nb_cat = cross_val_score(nb_cat, X, y, cv=cv, scoring="roc_auc").mean()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Mini-reto: ¿qué modelo para el equipo de reservas?

#### A7 · Mini-reto — Comparar los cuatro modelos

- **Qué hace:**
    - `train_test_split` separa el 80 % de las reservas para ajustar (`X_train`) y el 20 % para predecir (`X_test`), estratificado. `candidatos` es un diccionario con los cuatro modelos (los de antes más la regresión logística).
    - Para cada modelo: `cross_val_score(...).mean()` da el AUC en validación cruzada; `time.perf_counter()` marca el instante antes y después de `.fit(X_train, y_train)` y de `.predict_proba(X_test)`; las diferencias son los segundos de ajuste y de predicción.
    - `pd.DataFrame(filas).T.round(3)` los junta en una tabla (una fila por modelo). `recomendacion` es tu texto en 2 frases.
- **Qué debe salir:**
    - La tabla, tu `recomendacion` impresa y `✔ Mini-reto correcto` (los segundos varían un poco entre ejecuciones y entre ordenadores): `logistica` AUC 0,757 · `knn` 0,740 · `nb_gauss` 0,719 · `nb_cat` 0,742.
    - Tiempos orientativos: ajuste de 0,02–0,03 s en todos; predicción ≈ 0,006 s en los demás y **≈ 0,09–0,1 s en k-NN**.
- **Cómo leerlo:**
    - Observa que **k-NN tarda más en predecir que en «ajustar»** (al revés que casi todos los demás): tiene que comparar cada reserva nueva con todas las guardadas. En producción, con millones de reservas, eso pesa.
    - La logística gana en AUC (0,757) y además **se explica** (odds ratio); eso inclina la balanza a su favor.
- **Si falla:**
    - `Filas en el orden pedido` → falta `.T` o los modelos no están en el orden `logistica`, `knn`, `nb_gauss`, `nb_cat`.
    - `assert {"auc_cv", "segundos_ajuste", "segundos_predecir"} <= set(tabla.columns)` → alguna columna tiene otro nombre.
    - `AUC entre 0,6 y 0,9 en todos` → has medido el AUC con `predict` (clases) en lugar de con las probabilidades, o sobre el conjunto equivocado.
    - `Escribe una recomendación razonada` → `recomendacion` debe ser un texto de más de 40 caracteres.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué modelo tiene el mejor AUC y cuál el peor?* → Mejor **la logística (0,757)**; peor **`nb_gauss` (0,719)**. k-NN (0,740) y `nb_cat` (0,742) quedan en medio.
2. *¿Qué modelo tarda más en predecir y cuántas veces más que los demás?* → **k-NN**: ≈ 0,09–0,1 s frente a ≈ 0,006 s del resto, es decir, unas **15 veces más**.
3. *¿Qué modelo recomendarías y por qué? (razona con las tres columnas y la explicabilidad)* → **La regresión logística**: tiene el mejor AUC, ajusta y predice rápido (0,03 s y 0,005 s) y es la más explicable (cada coeficiente es un odds ratio). Para respuesta propia, vale cualquier modelo razonado con esas cuatro cosas.

</details>

## 5. Para reflexionar

1. ¿Por qué `k = 1` funciona tan mal aquí? ¿Qué pasaría con datos casi sin ruido?
2. Naive Bayes «se equivoca» al suponer independencia y aun así funciona. ¿Cuándo te fiarías menos de él?
3. Un cliente pregunta «¿por qué el modelo cree que esta reserva se cancelará?». ¿Con cuál de los cuatro modelos puedes responder mejor?
4. ¿Cómo cambia la respuesta si el modelo debe dar 1000 predicciones por segundo?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque con `k = 1` la predicción copia la clase de **un solo vecino**, y con datos ruidosos ese vecino se equivoca a menudo: el AUC (0,585) casi no mejora al azar (0,5). Con datos casi sin ruido, el vecino más cercano sí sería fiable y `k` pequeño funcionaría bien.
2. Cuando hay **variables muy relacionadas entre sí** (por ejemplo, precio, tarifa y temporada): su información se cuenta «doble» y las probabilidades salen exageradas. Y también cuando necesitas **probabilidades bien calibradas** (no solo ordenar reservas), porque las de Naive Bayes suelen ser demasiado extremas.
3. Con la **regresión logística**: cada coeficiente (odds ratio) dice cuánto sube o baja la probabilidad de cancelar al cambiar una variable. Naive Bayes da algo (qué variables empujan hacia cancelar) y k-NN casi nada: solo puede decir «se parece a estas reservas, que se cancelaron».
4. Aquí, con 4800 reservas guardadas, k-NN predice unas 12 000 por segundo (1200 en ≈ 0,1 s) y cumpliría; la logística y Naive Bayes, ≈ 200 000 por segundo (1200 en ≈ 0,006 s). Pero k-NN compara cada predicción con **todas las reservas guardadas**: con millones, dejaría de cumplir, y la logística y Naive Bayes seguirían valiendo.

</details>

## 6. Resumen: qué has aprendido
- **k-NN** predice por analogía; exige escalar y `k` se elige con validación cruzada; es lento al predecir.
- **Naive Bayes** multiplica probabilidades asumiendo independencia; es rápido y sencillo.
- Siempre se comparan **métrica + coste + explicabilidad**: el «mejor» modelo depende del contexto.